# Fossil & Dinosaur table analysis

Explore the `fossil` and `dinosaur` tables from the Mesozoica Postgres database.

**Prerequisites** (run once in the backend venv):

```bash
cd backend
source .venv/bin/activate
pip install pandas matplotlib seaborn jupyter ipykernel
```

**Database**: reads `DATABASE_URL` from `backend/.env` (same Railway Postgres URL used for local API dev).

In [99]:
from pathlib import Path

import pandas as pd
pd.set_option('display.max_columns', None)
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Resolve backend dir whether the notebook is opened from repo root or backend/
cwd = Path.cwd().resolve()
if (cwd / "app").is_dir():
    backend_dir = cwd
elif (cwd / "backend" / "app").is_dir():
    backend_dir = cwd / "backend"
else:
    backend_dir = cwd.parent if (cwd.parent / "app").is_dir() else cwd

load_dotenv(backend_dir / ".env", override=True)

import os

database_url = os.environ.get("DATABASE_URL", "")
if not database_url:
    raise ValueError("DATABASE_URL not set — add it to backend/.env")
if database_url.startswith("postgres://"):
    database_url = database_url.replace("postgres://", "postgresql://", 1)

engine = create_engine(database_url)
print(f"Connected via {backend_dir / '.env'}")

Connected via /Users/desiredewaele/git/mesozoica/backend/.env


In [100]:
fossil_df = pd.read_sql("SELECT * FROM fossil where llm_enriched = true", engine)
#dino_df = pd.read_sql("SELECT * FROM dinosaur", engine)

print(f"fossil rows: {len(fossil_df):,}")
#print(f"dinosaur rows: {len(dino_df):,}")

fossil rows: 2,661


In [101]:
fossil_df.llm_category.value_counts(dropna=False)

llm_category
unknown    1782
body        678
trace       201
Name: count, dtype: int64

In [102]:
fossil_df.llm_imp_preservation_quality.value_counts(dropna=False)

llm_imp_preservation_quality
good           973
moderate       520
excellent      369
poor           367
very_poor      223
exceptional    209
Name: count, dtype: int64

In [103]:
fossil_df[["llm_imp_category", "llm_imp_subcategory", "llm_imp_rock_type", "dinosaur_id", "llm_imp_completeness", "llm_imp_preservation_quality"]]

,llm_imp_category,llm_imp_subcategory,llm_imp_rock_type,dinosaur_id,llm_imp_completeness,llm_imp_preservation_quality
0,trace,gastroliths,phosphorite,60,trace_only,very_poor
1,body,pectoral_girdle,sandstone,60,fragmentary,good
2,body,vertebrae,sandstone,60,partial,good
3,trace,gastroliths,sandstone,60,trace_only,poor
4,body,skull,sandstone,60,partial,good
...,...,...,...,...,...,...
2656,trace,regurgitates,sandstone,53,trace_only,good
2657,trace,gastroliths,conglomerate,251,trace_only,good
2658,trace,regurgitates,claystone,114,trace_only,good
2659,body,vertebrae,sandstone,205,partial,good


In [104]:
fossil_df.llm_imp_category.value_counts(dropna=False)
pd.crosstab(fossil_df.llm_imp_subcategory, fossil_df.llm_imp_category)

llm_imp_category,body,trace
llm_imp_subcategory,,
bite_marks_and_feeding_traces,0,276
burrows_and_nesting_traces,0,133
coprolites,0,179
dermal_armour,85,0
eggs_and_embryos,73,0
footprints_and_trackways,0,192
forelimbs,95,0
gastroliths,0,173
hindlimbs,152,0


In [105]:
llm_fields = ["rock_type", "category", "subcategory", "preservation_quality", "completeness"]

for c in llm_fields:
    print(fossil_df[fossil_df.llm_enriched][f"llm_{c}"].value_counts())
    print(fossil_df[fossil_df.llm_enriched][f"llm_imp_{c}"].value_counts())

llm_rock_type
sandstone        877
unknown          857
mudstone         293
claystone        247
siltstone         98
marl              98
conglomerate      48
shale             47
limestone         33
chalk             28
coal              15
phosphorite        9
ironstone          4
siliciclastic      3
carbonate          2
tuff               1
other              1
Name: count, dtype: int64
llm_imp_rock_type
sandstone        921
mudstone         343
claystone        303
siltstone        159
marl             152
shale            114
conglomerate     104
limestone         85
chalk             78
other             65
coal              62
volcanic_ash      62
ironstone         56
evaporite         52
phosphorite       52
tuff              48
siliciclastic      3
carbonate          2
Name: count, dtype: int64
llm_category
unknown    1782
body        678
trace       201
Name: count, dtype: int64
llm_imp_category
body     1559
trace    1102
Name: count, dtype: int64
llm_subcategory
unknown

In [106]:
fossil_df.stratscale.value_counts()

stratscale
bed              1996
group of beds     392
formation          68
member             25
group               6
Name: count, dtype: int64

In [107]:
fossil_df.merge(dino_df, left_on="dinosaur_id", right_on="id", how="left").groupby("name").size().sort_values(ascending=False).head(60)

name
Triceratops            171
Allosaurus             147
Camarasaurus           141
Iguanodon              134
Tyrannosaurus           92
Stegosaurus             89
Massospondylus          84
Edmontosaurus           84
Apatosaurus             67
Diplodocus              66
Centrosaurus            56
Dromaeosaurus           48
Spinosaurus             47
Deinonychus             47
Giraffatitan            44
Carcharodontosaurus     44
Camptosaurus            43
Euskelosaurus           42
Ceratosaurus            38
Alamosaurus             38
Hypacrosaurus           37
Pachycephalosaurus      34
Corythosaurus           33
Albertosaurus           32
Dacentrurus             31
Coelophysis             31
Gallimimus              30
Cetiosaurus             30
Majungasaurus           30
Masiakasaurus           29
Chirostenotes           28
Lufengosaurus           28
Kentrosaurus            27
Kritosaurus             25
Aublysodon              25
Brontosaurus            23
Brachiosaurus          

In [7]:
fossil_df.geological_formation.value_counts().head(60)

geological_formation
Morrison                    695
Hell Creek                  202
Dinosaur Park               201
Lance                       137
Elliot                      136
Tendaguru                   104
Horseshoe Canyon             84
Maevarano                    54
Nemegt                       53
Two Medicine                 51
Oldman                       43
Cloverly                     42
Judith River                 41
Kirtland                     41
Lufeng                       39
Cedar Mountain               37
Bissekty                     32
Ojo Alamo                    30
Djadokhta                    30
Aguja                        29
Wessex                       29
Shaximiao                    29
Yixian                       28
Ferris                       26
Lourinhã                     26
Baruungoyot                  24
Frenchman                    22
Baynshire                    21
Arcillas de Morella          21
Weald Clay                   19
Candeleros         